# Solvformer: скачать и закодировать все шарды

Settings: Accelerator None (CPU, GPU-квота не тратится), Internet On.

Запуск: Save Version, Save & Run All. Output (`enc/*.npz`, ~1.5 ГБ) потом подключается к ноутбуку обучения
как датасет `hw2-enc`. Токенизатор и валидация берутся из репозитория, они те же, что на любой другой машине.

Код клонируется с GitHub в `/tmp`. Для приватного репозитория нужен секрет `GITHUB_TOKEN` (Add-ons, Secrets);
токен нигде не печатается и после клона убирается из `.git/config`. Сырые шарды скачиваются во временную папку,
в output идут только закодированные.

In [ ]:
import subprocess

REPO = 'dsnillisienko/hse-nlp'
CODE = '/tmp/solvformer'
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
except Exception:
    token = None
auth = f'x-access-token:{token}@' if token else ''
r = subprocess.run(['git', 'clone', '-q', f'https://{auth}github.com/{REPO}.git', CODE], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError('git clone failed: ' + (r.stderr.replace(token, '***') if token else r.stderr))
subprocess.run(['git', '-C', CODE, 'remote', 'set-url', 'origin', f'https://github.com/{REPO}.git'], check=True)
del token, auth
!git -C {CODE} log --oneline -3

In [ ]:
import os, shutil
DATA = '/tmp/solvformer-data'
OUT = '/kaggle/working/enc'
os.makedirs(DATA, exist_ok=True)
for f in ['tokenizer.json', 'val.jsonl']:
    shutil.copy(f'{CODE}/data/{f}', DATA)
!nproc; free -g | head -2; df -h /tmp /kaggle/working

## Скачивание (~1.6 ГБ, несколько минут)

In [ ]:
!cd {CODE} && python -m solvformer.prepare download --data {DATA} --shards 0-59

## Кодирование (~1 ч на CPU)

In [ ]:
!cd {CODE} && python -m solvformer.prepare encode --data {DATA} --out {OUT} --shards 0-59

## Проверка

In [ ]:
!ls {OUT}/*.npz | wc -l; du -sh {OUT}
import sys; sys.path.insert(0, CODE)
from solvformer.batches import Data
d = Data(sorted(f'{OUT}/{f}' for f in os.listdir(OUT) if f.endswith('.npz')))
print(f'{d.n:,} примеров, токенов q {d.q_len.sum():,} / c {d.c_len.sum():,}, мин. длины {d.q_len.min()} / {d.c_len.min()}')